# RNN for Sentiment Analysis - IMDB

In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

## pre-processing

### 1) Converting to lower case

In [7]:
df["review"]= df["review"].str.lower()

### 2) Removing the URLs

In [8]:
def remove_url(text):
    text = re.sub(r"http\S+","",text) #(pattern, replace,str)
    return text

df["review"] = df["review"].apply(remove_url)

### 3) Removing Punctuation

In [9]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text) #(pattern, replace,str)
    return text

df["review"] = df["review"].apply(remove_punctuation)

### 4) Remove HTML

In [10]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text) #(pattern, replace,str)
    return text

df["review"] = df["review"].apply(remove_html)

### 5) Removing the Stopwords

In [11]:
import nltk # natural language toolkit (imp)

nltk.download("punkt") # sentence tokenizer
nltk.download("punk_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\madhu\anaconda3\envs\pytorch\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Error loading punk_tab: Package 'punk_tab' not found in
[nltk_data]     index
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\madhu\anaconda3\envs\pytorch\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text

df["review"] = df["review"].apply(remove_stopwords)

## 6) Stemming

In [14]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

## 7) Encoding

In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"]= le.fit_transform(df["sentiment"])

In [16]:
y = df["sentiment"]

## 8) Vectorization

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## Dataset & Data Loaders

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

In [19]:
import torch
from torch.utils.data import TensorDataset, DataLoader


In [20]:
X_train = X_train.toarray() # converted the type to numpy array
X_test = X_test.toarray()

In [21]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()   # converting to data tensors
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

C:\Users\madhu\AppData\Local\Temp\ipykernel_12836\3677102300.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(y_train.values).float()   # converting to data tensors


In [22]:
train_loader = DataLoader(train_set, shuffle = True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [23]:
import torch.nn as nn
import torch.optim as optim

In [30]:
class RNN(nn.Module):
    def __init__(self,input_size, hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size= hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size,num_layers,batch_first = True) #batch first true becz that avoids continuous reshaping

        # Fully connected layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        # optional => shape (num of layers, batch_size, hidden_state)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x,h0)

        #1st value = hidden state of all the timestamps => (batch, seq_length, hidden size)
        #2nd value = final hidden state of last time stamp
        out = out[:, -1, :]

        out = self.fc(out)
        return out

In [31]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss() # binary crossentropy for binary class
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [32]:
# unsqueeze => to add a dimension
# squeeze => to reduce a dimension

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.2278376817703247
epoch = 2/10 and loss = 0.31106266379356384
epoch = 3/10 and loss = 0.2749684751033783
epoch = 4/10 and loss = 0.2487359493970871
epoch = 5/10 and loss = 0.17087775468826294
epoch = 6/10 and loss = 0.25264596939086914
epoch = 7/10 and loss = 0.20721492171287537
epoch = 8/10 and loss = 0.2903152406215668
epoch = 9/10 and loss = 0.4137236177921295
epoch = 10/10 and loss = 0.3668186068534851


In [33]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.65090249067259
